<a href="https://colab.research.google.com/github/ValueEdgeExplorer/looker-dashboard-site/blob/main/anchorfinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. INSTALLATION & SCRAPER CONFIGURATION
!pip install selenium pandas requests webdriver-manager
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get update
!apt-get install -y google-chrome-stable

import time
import pandas as pd
from io import StringIO
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from google.colab import files

def get_master_driver():
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=chrome_options)

def automate_anchor_collection(year):
    driver = get_master_driver()
    wait = WebDriverWait(driver, 25)

    master_url = f"https://www.chittorgarh.com/report/mainboard-ipos-anchor-investors/163/sme/?year={year}"
    print(f"Opening Master List for {year}: {master_url}")

    all_dfs = []
    ipo_links = []

    try:
        driver.get(master_url)

        page_num = 1
        while True:
            print(f"Collecting links from page {page_num}...")
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
            time.sleep(3)

            rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
            for row in rows:
                try:
                    cols = row.find_elements(By.TAG_NAME, "td")
                    if not cols or len(cols) < 1: continue
                    company_name = cols[0].text.strip()
                    link_element = cols[0].find_element(By.TAG_NAME, "a")
                    ipo_url = link_element.get_attribute('href')

                    if (company_name, ipo_url) not in ipo_links and ipo_url:
                        ipo_links.append((company_name, ipo_url))
                except: continue

            print(f"   Current unique IPO links: {len(ipo_links)}")

            try:
                # Enhanced pagination detection: try multiple strategies for 'Next'
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(2)

                # Try finding by text specifically or by the 'next' class in pagination
                next_found = False
                potential_next_selectors = [
                    "//a[contains(text(), 'Next') and not(contains(@class, 'disabled'))]",
                    "//li[contains(@class, 'next')]//a",
                    "//button[contains(text(), 'Next')]"
                ]

                for selector in potential_next_selectors:
                    btns = driver.find_elements(By.XPATH, selector)
                    if btns and btns[0].is_displayed():
                        # Verify it isn't a disabled parent li
                        parent = btns[0].find_element(By.XPATH, "..")
                        if "disabled" in btns[0].get_attribute("class") or "disabled" in parent.get_attribute("class"):
                            continue

                        driver.execute_script("arguments[0].click();", btns[0])
                        next_found = True
                        page_num += 1
                        time.sleep(5)
                        break

                if not next_found:
                    print("Reached the end of the list or no next button found.")
                    break
            except Exception as e:
                print(f"Pagination error: {e}")
                break

        print(f"\nTotal IPOs identified: {len(ipo_links)}")

        for company, url in ipo_links:
            print(f"Scraping Anchor Table for: {company}")
            driver.get(url)
            try:
                wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
                time.sleep(2)
                tables = pd.read_html(StringIO(driver.page_source))

                if len(tables) >= 4:
                    df = tables[3]
                    df = df[~df.iloc[:, 0].astype(str).str.contains('Total|S.No|Investor', case=False, na=False)]
                    df.insert(0, 'IPO Company', company)
                    all_dfs.append(df)
                else:
                    print(f"   ! Anchor table (Table 4) not found for {company}")
            except Exception as e:
                print(f"   ! Error processing {company}: {e}")
            time.sleep(1)

    finally:
        driver.quit()

    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        filename = f"SME_Anchors_{year}_Complete_Dataset.csv"
        final_df.to_csv(filename, index=False)
        print(f"\nExtraction Complete. Captured {len(final_df)} investor records.")
        display(final_df.head())
        files.download(filename)
        return final_df
    else:
        print("No data was collected.")
        return None

OK
Hit:1 http://dl.google.com/linux/chrome/deb stable InRelease
Hit:2 https://dl.google.com/linux/chrome-stable/deb stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 10.5 kB in 1s (8,081 B/s)
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: 

In [ ]:
# Run the scraper for the year 2025
automate_anchor_collection(2025)

Opening Master List for 2025: https://www.chittorgarh.com/report/mainboard-ipos-anchor-investors/163/sme/?year=2025
   Current unique IPO links: 25
   Current unique IPO links: 50
   Current unique IPO links: 75
   Current unique IPO links: 100
   Current unique IPO links: 125
   Current unique IPO links: 150
   Current unique IPO links: 175
   Current unique IPO links: 180
Reached the end of the list or no next button found.

Total IPOs identified: 180
Scraping Anchor Table for: Modern Diagnostic & Research Centre Ltd.
Scraping Anchor Table for: E to E Transportation Infrastructure Ltd.
Scraping Anchor Table for: Apollo Techno Industries Ltd.
Scraping Anchor Table for: Bai Kakaji Polymers Ltd.
Scraping Anchor Table for: Admach Systems Ltd.
Scraping Anchor Table for: Nanta Tech Ltd.
Scraping Anchor Table for: Dhara Rail Projects Ltd.
Scraping Anchor Table for: Shyam Dhani Industries Ltd.
Scraping Anchor Table for: Dachepalli Publishers Ltd.
Scraping Anchor Table for: EPW India Ltd.
Scr

,IPO Company,#,Anchor Investor,Group Entity,No. of Shares Allotted,Amount (Rs.cr.),% Allocated to each Anchor Investor,% Allotment of Issue,S.No.,% Allotment within Anchor Investor Portion,Investor Category,Subscription (times)
0,Modern Diagnostic & Research Centre Ltd.,1.0,AARTH AIF GROWTH FUND,AARTH,185600.0,1.67,15.98,4.53,NaN,NaN,NaN,NaN
1,Modern Diagnostic & Research Centre Ltd.,2.0,SB OPPORTUNITIES FUND II,SB OPPORTUNITIES FUND,164800.0,1.48,14.19,4.02,NaN,NaN,NaN,NaN
2,Modern Diagnostic & Research Centre Ltd.,3.0,SUNRISE INVESTMENT TRUST-SUNRISE INVESTMENT OP...,SUNRISE,136000.0,1.22,11.71,3.32,NaN,NaN,NaN,NaN
3,Modern Diagnostic & Research Centre Ltd.,4.0,360 ONE PRIME LTD.,360 ONE PRIME LTD.,113600.0,1.02,9.78,2.77,NaN,NaN,NaN,NaN
4,Modern Diagnostic & Research Centre Ltd.,5.0,NINE ALPS TRUST-NINE ALPS OPPORTUNITY FUND,NINE ALPS,113600.0,1.02,9.78,2.77,NaN,NaN,NaN,NaN


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,IPO Company,#,Anchor Investor,Group Entity,No. of Shares Allotted,Amount (Rs.cr.),% Allocated to each Anchor Investor,% Allotment of Issue,S.No.,% Allotment within Anchor Investor Portion,Investor Category,Subscription (times)
0,Modern Diagnostic & Research Centre Ltd.,1.0,AARTH AIF GROWTH FUND,AARTH,185600.0,1.67,15.98,4.53,NaN,NaN,NaN,NaN
1,Modern Diagnostic & Research Centre Ltd.,2.0,SB OPPORTUNITIES FUND II,SB OPPORTUNITIES FUND,164800.0,1.48,14.19,4.02,NaN,NaN,NaN,NaN
2,Modern Diagnostic & Research Centre Ltd.,3.0,SUNRISE INVESTMENT TRUST-SUNRISE INVESTMENT OP...,SUNRISE,136000.0,1.22,11.71,3.32,NaN,NaN,NaN,NaN
3,Modern Diagnostic & Research Centre Ltd.,4.0,360 ONE PRIME LTD.,360 ONE PRIME LTD.,113600.0,1.02,9.78,2.77,NaN,NaN,NaN,NaN
4,Modern Diagnostic & Research Centre Ltd.,5.0,NINE ALPS TRUST-NINE ALPS OPPORTUNITY FUND,NINE ALPS,113600.0,1.02,9.78,2.77,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1439,Leo Dryfruits & Spices Trading Ltd.,NaN,SAINT CAPITAL FUND,SAINT CAPITAL,194000.0,1.01,NaN,4.02,3.0,14.65,NaN,NaN
1440,Leo Dryfruits & Spices Trading Ltd.,NaN,VIKASA CAPITAL INC.,VIKASA,290000.0,1.51,NaN,6.00,4.0,21.90,NaN,NaN
1441,Leo Dryfruits & Spices Trading Ltd.,NaN,CHANAKYA OPPORTUNITIES FUND,CHANAKYA,194000.0,1.01,NaN,4.02,5.0,14.65,NaN,NaN
1442,Leo Dryfruits & Spices Trading Ltd.,NaN,BEACON STONE CAPITAL VCC-BEACON STONE I,BEACON STONE CAPITAL,258000.0,1.34,NaN,5.34,6.0,19.49,NaN,NaN


I see the problem, its fetching data from the first table it saw for that particular IPO but imo the data i want is stored in 4th table and that's how i extracted 4th table from all the ipos in which anchor is avialable

In [ ]:
df_2022 = automate_anchor_collection(2022)

Opening Master List for 2022: https://www.chittorgarh.com/report/mainboard-ipos-anchor-investors/163/sme/?year=2022
   Current unique IPO links: 12
Reached the end of the list or no next button found.

Total IPOs identified: 12
Scraping Anchor Table for: Anlon Technology Solutions Ltd.
Scraping Anchor Table for: Droneacharya Aerial Innovations Ltd.
Scraping Anchor Table for: All E Technologies Ltd.
Scraping Anchor Table for: Vital Chemtech Ltd.
Scraping Anchor Table for: Phantom Digital Effects Ltd.
Scraping Anchor Table for: Frog Cellsat Ltd.
Scraping Anchor Table for: Cyber Media Research & Services Ltd.
Scraping Anchor Table for: Concord Control Systems Ltd.
Scraping Anchor Table for: Annapurna Swadisht Ltd.
Scraping Anchor Table for: Krishna Defence & Allied Industries Ltd.
Scraping Anchor Table for: P.E.Analytics Ltd.
Scraping Anchor Table for: KN Agri Resources Ltd.

Extraction Complete. Captured 46 investor records.


,IPO Company,S.No.,Anchor Investor,Group Entity,No. of Shares Allotted,Amount (Rs.cr.),% Allotment within Anchor Investor Portion,% Allotment of Issue
0,Anlon Technology Solutions Ltd.,1.0,NAV CAPITAL VCC,NAV CAPITAL,325200,3.25,76.34,21.68
1,Anlon Technology Solutions Ltd.,2.0,ELARA INDIA OPPORTUNITIES FUND LTD.,ELARA,100800,1.01,23.66,6.72
2,Anlon Technology Solutions Ltd.,NaN,NaN,NaN,426000,4.26,100.00,28.40
3,Droneacharya Aerial Innovations Ltd.,1.0,AEGIS INVESTMENT FUND,AEGIS INVESTMENT FUND,372000,2.01,15.98,5.91
4,Droneacharya Aerial Innovations Ltd.,2.0,MAVEN INDIA FUND,MAVEN,740000,4.00,31.79,11.76


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_2026 = automate_anchor_collection(2026)

Opening Master List for 2026: https://www.chittorgarh.com/report/mainboard-ipos-anchor-investors/163/sme/?year=2026
   Current unique IPO links: 25
   Current unique IPO links: 29
Reached the end of the list or no next button found.

Total IPOs identified: 29
Scraping Anchor Table for: Safety Controls & Devices Ltd.
Scraping Anchor Table for: Emiac Technologies Ltd.
Scraping Anchor Table for: Vivid Electromech Ltd.
Scraping Anchor Table for: Highness Microelectronics Ltd.
Scraping Anchor Table for: Tipco Engineering India Ltd.
Scraping Anchor Table for: Novus Loyalty Ltd.
Scraping Anchor Table for: Apsis Aerocom Ltd.
Scraping Anchor Table for: Striders Impex Ltd.
Scraping Anchor Table for: Yaap Digital Ltd.
Scraping Anchor Table for: Mobilise App Lab Ltd.
Scraping Anchor Table for: Accord Transformer & Switchgear Ltd.
Scraping Anchor Table for: Manilam Industries India Ltd.
Scraping Anchor Table for: Fractal Industries Ltd.
Scraping Anchor Table for: Marushika Technology Ltd.
Scraping 

,IPO Company,#,Anchor,Group Entity,Shares Allotted,Amt (₹ cr.),% Allocated,% Allotment of Issue,Anchor Investor,No. of Shares Allotted,Amount (Rs.cr.),% Allocated to each Anchor Investor
0,Safety Controls & Devices Ltd.,1.0,SHINE STAR BUILD-CAP PVT.LTD.,SHINE STAR,750400.0,6.00,47.37,12.51,NaN,NaN,NaN,NaN
1,Safety Controls & Devices Ltd.,2.0,VBCUBE VENTURES FUND,VBCUBE,251200.0,2.01,15.86,4.19,NaN,NaN,NaN,NaN
2,Safety Controls & Devices Ltd.,3.0,VBCUBE VENTURES FUND II,-,251200.0,2.01,15.86,4.19,NaN,NaN,NaN,NaN
3,Safety Controls & Devices Ltd.,4.0,PESB ALPHA FUND,PESB CAPITAL,126400.0,1.01,7.98,2.11,NaN,NaN,NaN,NaN
4,Safety Controls & Devices Ltd.,5.0,UPSURGE OPPORTUNITIES FUND I,UPSURGE,204800.0,1.64,12.93,3.41,NaN,NaN,NaN,NaN


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_2023 = automate_anchor_collection(2023)

Opening Master List for 2023: https://www.chittorgarh.com/report/mainboard-ipos-anchor-investors/163/sme/?year=2023
   Current unique IPO links: 25
   Current unique IPO links: 50
   Current unique IPO links: 65
Reached the end of the list or no next button found.

Total IPOs identified: 65
Scraping Anchor Table for: Kaushalya Logistics Ltd.
Scraping Anchor Table for: Kay Cee Energy & Infra Ltd.
Scraping Anchor Table for: Akanksha Power & Infrastructure Ltd.
Scraping Anchor Table for: Shri Balaji Valve Components Ltd.
Scraping Anchor Table for: Supreme Power Equipment Ltd.
Scraping Anchor Table for: Trident Techlabs Ltd.
Scraping Anchor Table for: Shanti Spintex Ltd.
Scraping Anchor Table for: Siyaram Recycling Industries Ltd.
Scraping Anchor Table for: S J Logistics (India) Ltd.
Scraping Anchor Table for: Accent Microcell Ltd.
Scraping Anchor Table for: Net Avenue Technologies Ltd.
Scraping Anchor Table for: AMIC Forging Ltd.
Scraping Anchor Table for: Deepak Chemtex Ltd.
Scraping Anc

,IPO Company,S.No.,Anchor Investor,Group Entity,No. of Shares Allotted,Amount (Rs.cr.),% Allotment within Anchor Investor Portion,% Allotment of Issue,Investor Category,Subscription (times),#,% Allocated to each Anchor Investor,0,1
0,Kaushalya Logistics Ltd.,1.0,NEOMILE GROWTH FUND,NEOMILE,664000.0,4.98,49.76,13.61,NaN,NaN,NaN,NaN,NaN,NaN
1,Kaushalya Logistics Ltd.,2.0,RAJASTHAN GLOBAL SECURITIES PVT.LTD.,RAJASTHAN GLOBAL SECURITIES,134400.0,1.01,10.07,2.75,NaN,NaN,NaN,NaN,NaN,NaN
2,Kaushalya Logistics Ltd.,3.0,LRSD SECURITIES PVT.LTD.,LRSD SECURITIES,134400.0,1.01,10.07,2.75,NaN,NaN,NaN,NaN,NaN,NaN
3,Kaushalya Logistics Ltd.,4.0,SAINT CAPITAL FUND,SAINT CAPITAL,267200.0,2.00,20.02,5.48,NaN,NaN,NaN,NaN,NaN,NaN
4,Kaushalya Logistics Ltd.,5.0,LC RADIANCE FUND VCC,LC RADIANCE,134400.0,1.01,10.07,2.75,NaN,NaN,NaN,NaN,NaN,NaN


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_2024 = automate_anchor_collection(2024)

Opening Master List for 2024: https://www.chittorgarh.com/report/mainboard-ipos-anchor-investors/163/sme/?year=2024
   Current unique IPO links: 25
   Current unique IPO links: 50
   Current unique IPO links: 75
   Current unique IPO links: 100
   Current unique IPO links: 125
   Current unique IPO links: 150
   Current unique IPO links: 152
Reached the end of the list or no next button found.

Total IPOs identified: 152
Scraping Anchor Table for: Technichem Organics Ltd.
Scraping Anchor Table for: Anya Polytech & Fertilizers Ltd.
Scraping Anchor Table for: Identical Brains Studios Ltd.
Scraping Anchor Table for: NACDAC Infrastructure Ltd.
Scraping Anchor Table for: Yash Highvoltage Ltd.
Scraping Anchor Table for: Purple United Sales Ltd.
Scraping Anchor Table for: Jungle Camps India Ltd.
Scraping Anchor Table for: Toss The Coin Ltd.
Scraping Anchor Table for: Dhanlaxmi Crop Science Ltd.
Scraping Anchor Table for: Emerald Tyre Manufacturers Ltd.
Scraping Anchor Table for: Nisus Finance

,IPO Company,S.No.,Anchor Investor,Group Entity,No. of Shares Allotted,Amount (Rs.cr.),% Allotment within Anchor Investor Portion,% Allotment of Issue,Investor Category,Subscription (times)
0,Technichem Organics Ltd.,1.0,VIKASA CAPITAL INC.,VIKASA,194000.0,1.07,14.95,4.23,NaN,NaN
1,Technichem Organics Ltd.,2.0,NEXT ORBIT GROWTH FUND III,NEXT ORBIT,182000.0,1.00,14.02,3.97,NaN,NaN
2,Technichem Organics Ltd.,3.0,ABUNDANTIA CAPITAL VCC-ABUNDANTIA CAPITAL III,ABUNDANTIA,194000.0,1.07,14.95,4.23,NaN,NaN
3,Technichem Organics Ltd.,4.0,MINT FOCUSED GROWTH FUND PCC-CELL 1,MINT FOCUSED,182000.0,1.00,14.02,3.97,NaN,NaN
4,Technichem Organics Ltd.,5.0,ARNESTA GLOBAL OPPORTUNITIES FUND PCC-ARNESTA ...,ARNESTA,182000.0,1.00,14.02,3.97,NaN,NaN


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>